In [ ]:
from ngsxditto import *
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.15))

In [ ]:
dt = 0.05
t = Parameter(0)
starting_levelset = (5*x**2 + y**2)**(1/2) - 2.0/3.0
transport = ExplicitDGTransport(mesh, dt=dt, order=2, compile=False)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)

In [ ]:
fluid1_params = FluidParameters(viscosity=1e-2)
fluid2_params = FluidParameters(viscosity=1e-2)

mean_curvature = MeanCurvatureSolver(mesh, order=2, lset=levelset)
mean_curvature.Step()
fluid = TwoPhaseTaylorHood(mesh, fluid1_params=fluid1_params, fluid2_params=fluid2_params, lset=levelset, surface_tension=mean_curvature.H, dt=dt, order=3, ghost_stab=0.01)
fluid.Initialize(dirichlet={".*": CF((0, 0))})
fluid.ValidateStep()
sol = fluid.SolveStokes()
u1, p1, u2, p2 = sol.components
ngw.Draw(IfPos(levelset.lsetp1, u2, u1), mesh, "u")

In [ ]:
velocity_extension = DiffusionBasedVelocityExtension(levelset)

velocity_extension.SetRhs(fluid.gfu.components[0])
levelset.transport.SetWind(velocity_extension.field)

end_time = 2

time_loop = TimeLoop(time=t, dt=dt, end_time=end_time)

sphericity = SphericityDiagram(levelset, time=t, name="sphericity")

cf_neg = Norm(fluid.gfu.components[0])
cf_pos = Norm(fluid.gfu.components[2])
animation = UnfittedNGSWebguiPlot(levelset, cf_neg=cf_neg, cf_pos=cf_pos,
                                  order=fluid.order, time=t, end_time=end_time,
                                  name="animation", min=0, max=0.2, autoscale=False)
def SetVelExtRhs():
    velocity_extension.SetRhs(fluid.gfu.components[0])

def SetCFNeg():
    animation.SetCFNeg(Norm(fluid.gfu.components[0]))
def SetCFPos():
    animation.SetCFPos(Norm(fluid.gfu.components[2]))



time_loop.Register(fluid, name="moving stokes")
time_loop.Register(SetVelExtRhs, name="update fluid")
time_loop.Register(velocity_extension, name="vel ext.")
time_loop.Register(levelset, name="levelset")
time_loop.Register(mean_curvature, name="mean curvature")

time_loop.Register(SetCFNeg, name="update neg values")
time_loop.Register(SetCFPos, name="update pos values")

time_loop.Register(sphericity, name="sphericity")
time_loop.Register(animation, name="animation")
time_loop()
